In [4]:
import pandas as pd
from corpus import build_corpus
from datasets import load_dataset
import tiktoken
from chunking import chunk_corpus
from embedding import get_embeddings

In [3]:
df=pd.read_parquet("../data/processed/df_val_enriched.parquet")


In [39]:
corpus=build_corpus(df)

In [40]:
print(list(corpus.items())[:5])
print("length of the global corpus : ",len(corpus))
print("average length of the documents in the corpus : ",sum(len(x) for x in corpus.values())/len(corpus))

[('Ed Wood (film)', "Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.  Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast."), ('Scott Derrickson', 'Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."'), ('Woodson, Arkansas', 'Woodson is a census-designated place (CDP) in Pulaski County, Arkansas, in the United States.  Its population was 403 at the 2010 census.  It is part of the Little Rock–North

In [41]:
enc=tiktoken.get_encoding("cl100k_base")
length_encoded=[]
for text in corpus.values():
    text_encoded=enc.encode(text)
    length_encoded.append(len(text_encoded))

pd.Series(length_encoded).describe()

    

count    66581.000000
mean       125.304606
std         71.231108
min          8.000000
25%         77.000000
50%        112.000000
75%        158.000000
max       1979.000000
dtype: float64

In [42]:
m=(pd.Series(length_encoded)>300).sum()
print("Number of documents with length greater than 300 : ",m)

Number of documents with length greater than 300 :  1541


In [20]:
df

,id,question,answer,type,level,supporting_facts,context,num_gold_paragraphs,num_titles_in_question,number_sf_in_context
0,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same ...,yes,comparison,hard,"{'title': ['Scott Derrickson', 'Ed Wood'], 'se...","{'title': ['Ed Wood (film)', 'Scott Derrickson...",2,2,2
1,5a8c7595554299585d9e36b6,What government position was held by the woman...,Chief of Protocol,bridge,hard,"{'title': ['Kiss and Tell (1945 film)', 'Shirl...","{'title': ['Meet Corliss Archer', 'Shirley Tem...",2,0,2
2,5a85ea095542994775f606a8,"What science fantasy young adult series, told ...",Animorphs,bridge,hard,"{'title': ['The Hork-Bajir Chronicles', 'The H...","{'title': ['Andre Norton Award', 'Victoria Han...",2,0,2
3,5adbf0a255429947ff17385a,Are the Laleli Mosque and Esma Sultan Mansion ...,no,comparison,hard,"{'title': ['Laleli Mosque', 'Esma Sultan Mansi...",{'title': ['Esma Sultan (daughter of Abdülaziz...,2,2,2
4,5a8e3ea95542995a26add48d,"The director of the romantic comedy ""Big Stone...","Greenwich Village, New York City",bridge,hard,"{'title': ['Big Stone Gap (film)', 'Adriana Tr...",{'title': ['Just Another Romantic Wrestling Co...,2,0,2
...,...,...,...,...,...,...,...,...,...,...
7400,5ab92307554299753720f72d,What Pakistani actor and writer from Islamabad...,Yasir Hussain,bridge,hard,"{'title': ['Coke Kahani', 'Coke Kahani', 'Yasi...","{'title': ['Coke Kahani', 'Adnan Siddiqui', 'F...",2,1,2
7401,5abba3b1554299642a094aed,Are both Volvic and Canfield's Diet Chocolate ...,no,comparison,hard,"{'title': ['Volvic (mineral water)', 'Canfield...","{'title': ['ZERO bar', 'Pseudomonas mandelii',...",2,1,2
7402,5a8173fa554299260e20a28e,Are Billy and Barak both breeds of scenthound?...,yes,comparison,hard,"{'title': ['Bosnian Coarse-haired Hound', 'Bil...","{'title': ['Styrian Coarse-haired Hound', 'Bil...",2,1,2
7403,5a8caf1d554299585d9e3720,Were both of the following rock groups formed ...,yes,comparison,hard,"{'title': ['Dig (band)', 'Thinking Fellers Uni...","{'title': ['Strangers from the Universe', 'Mot...",2,1,2


In [47]:
pilot=df.groupby(['type','num_titles_in_question']).apply(lambda g : g.sample(n=25,random_state=4))

In [48]:
pilot=pilot.reset_index(drop=False)
print(pilot.index)
print(pilot.columns)
pilot

RangeIndex(start=0, stop=150, step=1)
Index(['type', 'num_titles_in_question', 'level_2', 'id', 'question', 'answer',
       'level', 'supporting_facts', 'context', 'num_gold_paragraphs',
       'number_sf_in_context'],
      dtype='str')


,type,num_titles_in_question,level_2,id,question,answer,level,supporting_facts,context,num_gold_paragraphs,number_sf_in_context
0,bridge,0,6099,5ade66f055429939a52fe8a0,Easter Airways' head office is located at whic...,Humberside Airport,hard,"{'title': ['Eastern Airways', 'Humberside Airp...","{'title': ['Manchester Interchange', 'Islay Ai...",2,2
1,bridge,0,1464,5a7f924c55429969796c1aba,Which Star of British-Canadian-American satiri...,Eric Banadinović,hard,"{'title': ['Special Correspondents', 'Eric Ban...","{'title': ['John Walker (Australian actor)', '...",2,2
2,bridge,0,946,5adfda5255429942ec259b5e,What county in Minnesota holds a 202 lb slab o...,Douglas,hard,"{'title': ['Kensington, Minnesota', 'Kensingto...","{'title': ['Bridge No. 3355-Kathio Township', ...",2,2
3,bridge,0,6589,5ae4e29855429908b63264a9,Where does the team that Flavio de Jesús Santo...,city of Culiacán,hard,"{'title': ['Flavio Santos', 'Dorados de Sinalo...","{'title': ['Adelaide Comets FC', 'Sheffield We...",2,2
4,bridge,0,888,5ab8ed305542991b5579f046,Keene is an unincorporated community in Wabaun...,The United States of America,hard,"{'title': ['Keene, Kansas', 'United States'], ...","{'title': ['United States', 'Administrative di...",2,2
...,...,...,...,...,...,...,...,...,...,...,...
145,comparison,2,6141,5ab2b7405542991669774069,Have James Schuyler and H. G. Wells both publi...,yes,hard,"{'title': ['James Schuyler', 'H. G. Wells'], '...","{'title': ['H. G. Wells', 'Pierre Martory', 'C...",2,2
146,comparison,2,3812,5a84d2ea5542992a431d1aa8,What career titles do Ed Wood and Fernand Rive...,"actor, writer, producer, and director",hard,"{'title': ['Ed Wood', 'Fernand Rivers'], 'sent...","{'title': ['Ed Wood', 'The Two Boys (1936 film...",2,2
147,comparison,2,1491,5a8dad37554299068b959d94,Both Phaedranassa and Mitella are considered t...,genus,hard,"{'title': ['Phaedranassa', 'Mitella'], 'sent_i...","{'title': ['Asiatosuchus', 'Ardipithecus kadab...",2,2
148,comparison,2,1227,5a79e3925542994bb9457130,Name three best sellers from Arthur Hailey tha...,"Hotel"" (1965), ""Airport"" (1968), ""Wheels"" (1971)",hard,"{'title': ['Arthur Hailey', 'Donna Tartt'], 's...","{'title': ['Arthur Hailey', 'Airport (1970 fil...",2,2


In [50]:
pilot=pilot.drop(columns=["level_2"])
pilot

,type,num_titles_in_question,id,question,answer,level,supporting_facts,context,num_gold_paragraphs,number_sf_in_context
0,bridge,0,5ade66f055429939a52fe8a0,Easter Airways' head office is located at whic...,Humberside Airport,hard,"{'title': ['Eastern Airways', 'Humberside Airp...","{'title': ['Manchester Interchange', 'Islay Ai...",2,2
1,bridge,0,5a7f924c55429969796c1aba,Which Star of British-Canadian-American satiri...,Eric Banadinović,hard,"{'title': ['Special Correspondents', 'Eric Ban...","{'title': ['John Walker (Australian actor)', '...",2,2
2,bridge,0,5adfda5255429942ec259b5e,What county in Minnesota holds a 202 lb slab o...,Douglas,hard,"{'title': ['Kensington, Minnesota', 'Kensingto...","{'title': ['Bridge No. 3355-Kathio Township', ...",2,2
3,bridge,0,5ae4e29855429908b63264a9,Where does the team that Flavio de Jesús Santo...,city of Culiacán,hard,"{'title': ['Flavio Santos', 'Dorados de Sinalo...","{'title': ['Adelaide Comets FC', 'Sheffield We...",2,2
4,bridge,0,5ab8ed305542991b5579f046,Keene is an unincorporated community in Wabaun...,The United States of America,hard,"{'title': ['Keene, Kansas', 'United States'], ...","{'title': ['United States', 'Administrative di...",2,2
...,...,...,...,...,...,...,...,...,...,...
145,comparison,2,5ab2b7405542991669774069,Have James Schuyler and H. G. Wells both publi...,yes,hard,"{'title': ['James Schuyler', 'H. G. Wells'], '...","{'title': ['H. G. Wells', 'Pierre Martory', 'C...",2,2
146,comparison,2,5a84d2ea5542992a431d1aa8,What career titles do Ed Wood and Fernand Rive...,"actor, writer, producer, and director",hard,"{'title': ['Ed Wood', 'Fernand Rivers'], 'sent...","{'title': ['Ed Wood', 'The Two Boys (1936 film...",2,2
147,comparison,2,5a8dad37554299068b959d94,Both Phaedranassa and Mitella are considered t...,genus,hard,"{'title': ['Phaedranassa', 'Mitella'], 'sent_i...","{'title': ['Asiatosuchus', 'Ardipithecus kadab...",2,2
148,comparison,2,5a79e3925542994bb9457130,Name three best sellers from Arthur Hailey tha...,"Hotel"" (1965), ""Airport"" (1968), ""Wheels"" (1971)",hard,"{'title': ['Arthur Hailey', 'Donna Tartt'], 's...","{'title': ['Arthur Hailey', 'Airport (1970 fil...",2,2


In [52]:
final_corpus=chunk_corpus(corpus,threshold=300)

In [55]:
print("Shape of final corpus:", final_corpus.shape)
final_corpus

Shape of final corpus: (68211, 4)


,chunk_id,title,chunk_index,text
0,Ed Wood (film)_0,Ed Wood (film),0,Ed Wood is a 1994 American biographical period...
1,Scott Derrickson_0,Scott Derrickson,0,"Scott Derrickson (born July 16, 1966) is an Am..."
2,"Woodson, Arkansas_0","Woodson, Arkansas",0,Woodson is a census-designated place (CDP) in ...
3,Tyler Bates_0,Tyler Bates,0,"Tyler Bates (born June 5, 1965) is an American..."
4,Ed Wood_0,Ed Wood,0,"Edward Davis Wood Jr. (October 10, 1924 – Dece..."
...,...,...,...,...
68206,Analog Devices_0,Analog Devices,0,"Analog Devices, Inc., also known as ADI or Ana..."
68207,Zet (hardware)_0,Zet (hardware),0,Zet is a clone x86 processor where its machine...
68208,Xetal_0,Xetal,0,Xetal is the name of a family of non commercia...
68209,XAP processor_0,XAP processor,0,The XAP processor is a RISC processor architec...


In [56]:
pilot.to_parquet("../data/processed/pilot.parquet")
final_corpus.to_parquet("../data/processed/final_corpus.parquet")

In [5]:
final_corpus=pd.read_parquet("../data/processed/final_corpus.parquet")

In [7]:
final_corpus
final_corpus_embed=get_embeddings(final_corpus)

Batches:   0%|          | 0/2132 [00:00<?, ?it/s]

In [8]:
final_corpus_embed

,chunk_id,title,chunk_index,text,embeddings
0,Ed Wood (film)_0,Ed Wood (film),0,Ed Wood is a 1994 American biographical period...,"[-0.0051703453, 0.019602325, -0.031824153, 0.0..."
1,Scott Derrickson_0,Scott Derrickson,0,"Scott Derrickson (born July 16, 1966) is an Am...","[0.002950991, -0.022987213, -0.005424717, -0.0..."
2,"Woodson, Arkansas_0","Woodson, Arkansas",0,Woodson is a census-designated place (CDP) in ...,"[0.023222366, 0.016508676, -0.05767146, -0.033..."
3,Tyler Bates_0,Tyler Bates,0,"Tyler Bates (born June 5, 1965) is an American...","[0.0015271629, -0.00393534, -0.038948067, 0.07..."
4,Ed Wood_0,Ed Wood,0,"Edward Davis Wood Jr. (October 10, 1924 – Dece...","[0.011546341, -0.021726314, -0.023861984, 0.00..."
...,...,...,...,...,...
68206,Analog Devices_0,Analog Devices,0,"Analog Devices, Inc., also known as ADI or Ana...","[-0.002939147, -0.01783917, -0.043529715, 0.02..."
68207,Zet (hardware)_0,Zet (hardware),0,Zet is a clone x86 processor where its machine...,"[-0.0324406, -0.013010039, -0.026541445, 0.032..."
68208,Xetal_0,Xetal,0,Xetal is the name of a family of non commercia...,"[-0.064489804, 0.008572435, -0.019860506, 0.07..."
68209,XAP processor_0,XAP processor,0,The XAP processor is a RISC processor architec...,"[-0.063907765, -0.011429355, -0.053076703, 0.0..."


In [9]:
final_corpus_embed.to_parquet("../data/processed/final_corpus_embed.parquet")